# Step 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Step 2: Import Libraries

In [2]:
import os
import sqlite3
import pandas as pd

# Step 3: Set Input and Output Paths

In [3]:
processed_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/data/processed"

output_path = "/content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline"

os.makedirs(output_path, exist_ok=True)

partner_master_file = os.path.join(processed_path, "partner_master_clean.csv")
partner_pricing_file = os.path.join(processed_path, "partner_pricing_clean.csv")
partner_metrics_file = os.path.join(processed_path, "partner_metrics_clean.csv")

# Step 4: Load the Cleaned CSV Files

In [4]:
partner_master = pd.read_csv(partner_master_file)
partner_pricing = pd.read_csv(partner_pricing_file)
partner_metrics = pd.read_csv(partner_metrics_file)

print("partner_master:", partner_master.shape)
print("partner_pricing:", partner_pricing.shape)
print("partner_metrics:", partner_metrics.shape)

partner_master: (5000, 20)
partner_pricing: (5000, 16)
partner_metrics: (120000, 20)


# Step 5: Check Columns Before Loading to SQLite

In [5]:
print("partner_master columns:")
print(partner_master.columns.tolist())

print("\npartner_pricing columns:")
print(partner_pricing.columns.tolist())

print("\npartner_metrics columns:")
print(partner_metrics.columns.tolist())

partner_master columns:
['partner_id', 'partner_name', 'partner_type', 'industry_vertical', 'partner_size', 'partner_status', 'country', 'state', 'city', 'region', 'integration_type', 'sales_channel', 'risk_tier', 'kyc_status', 'pci_level', 'onboarding_date', 'relationship_manager_region', 'synthetic_record_flag', 'industry', 'status']

partner_pricing columns:
['assignment_id', 'partner_id', 'pricing_plan_id', 'recommended_pricing_plan_id', 'effective_date', 'expiration_date', 'review_due_date', 'negotiated_bps', 'negotiated_per_txn_fee_usd', 'monthly_minimum_fee_usd', 'exception_flag', 'exception_type', 'approval_status', 'approved_by_role', 'pricing_tier', 'contract_status']

partner_metrics columns:
['partner_id', 'month', 'txn_count', 'payment_volume_usd', 'avg_ticket_usd', 'txn_growth_pct', 'volume_growth_pct', 'chargeback_rate', 'auth_approval_rate', 'active_merchants', 'refunds_usd', 'net_revenue_usd', 'gross_margin_usd', 'gross_margin_rate', 'metric_id', 'transaction_month', '

# Step 6: Create SQLite Database

In [6]:
db_path = os.path.join(output_path, "partnerlens.db")

conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("SQLite database created at:", db_path)

SQLite database created at: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline/partnerlens.db


# Step 7: Load DataFrames into SQLite Tables

In [7]:
partner_master.to_sql("partner_master", conn, if_exists="replace", index=False)
partner_pricing.to_sql("partner_pricing", conn, if_exists="replace", index=False)
partner_metrics.to_sql("partner_metrics", conn, if_exists="replace", index=False)

print("Tables loaded successfully.")

Tables loaded successfully.


# Step 8: Confirm Tables Were Created

In [8]:
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

tables

,name
0,partner_master
1,partner_pricing
2,partner_metrics


# Step 9: Check Table Row Counts

In [9]:
row_count_query = """
SELECT 'partner_master' AS table_name, COUNT(*) AS row_count FROM partner_master
UNION ALL
SELECT 'partner_pricing' AS table_name, COUNT(*) AS row_count FROM partner_pricing
UNION ALL
SELECT 'partner_metrics' AS table_name, COUNT(*) AS row_count FROM partner_metrics;
"""

row_counts = pd.read_sql_query(row_count_query, conn)
row_counts

,table_name,row_count
0,partner_master,5000
1,partner_pricing,5000
2,partner_metrics,120000


In [12]:
row_counts.to_csv(
    os.path.join(output_path, "phase4_table_row_counts.csv"),
    index=False
)

# Step 10: Create a Helper Function to Run SQL

In [10]:
def run_sql(query):
    return pd.read_sql_query(query, conn)

Test it:

In [11]:
run_sql("SELECT * FROM partner_master LIMIT 5;")

,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,country,state,city,region,integration_type,sales_channel,risk_tier,kyc_status,pci_level,onboarding_date,relationship_manager_region,synthetic_record_flag,industry,status
0,P000001,NorthStar Commerce 0001,Technology Platform,Travel,SMB,Suspended,US,NY,New York,Northeast,API,Direct,High,Approved,Level 3,2019-06-15,Northeast,1,Travel,Suspended
1,P000002,Ironwood Market 0002,ISV,SaaS,Mid-Market,Pilot,US,VA,Norfolk,South,SDK,Direct,Low,Approved,Level 4,2020-02-13,South,1,SaaS,Pilot
2,P000003,Evergreen Commerce 0003,Technology Platform,Professional Services,SMB,Active,US,CO,Colorado Springs,West,API,Direct,Low,Approved,Level 3,2023-03-12,West,1,Professional Services,Active
3,P000004,Summit Platform 0004,ISO/Reseller,Gaming,SMB,Active,US,GA,Augusta,South,API,Referral,Medium,Approved,Level 4,2020-02-13,South,1,Gaming,Active
4,P000005,Prairie PayTech 0005,Merchant Acquirer,Healthcare,SMB,Active,US,CA,San Diego,West,API,Direct,High,Approved,Level 4,2022-03-09,West,1,Healthcare,Active


# Step 11: Write Baseline SQL Query 1

**Business Question 1**

Show me technology partners in Arizona with transaction growth above 20%.

In [13]:
q1_sql = """
SELECT
    pm.partner_id,
    pm.partner_name,
    pm.partner_type,
    pm.industry_vertical,
    pm.state,
    pm.partner_status,
    mt.transaction_month,
    mt.growth_pct
FROM partner_master pm
JOIN partner_metrics mt
    ON pm.partner_id = mt.partner_id
WHERE pm.partner_type LIKE '%Technology%'
  AND pm.state = 'AZ'
  AND mt.growth_pct > 20
ORDER BY mt.growth_pct DESC;
"""

q1_result = run_sql(q1_sql)
q1_result.head()

,partner_id,partner_name,partner_type,industry_vertical,state,partner_status,transaction_month,growth_pct
0,P004892,Summit Connect 4892,Technology Platform,Travel,AZ,Active,2024-12-01,55.72
1,P003960,Clearwater Processing 3960,Technology Platform,Hospitality,AZ,Pilot,2024-12-01,43.32
2,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2024-11-01,41.59
3,P004612,Sunrise Connect 4612,Technology Platform,Travel,AZ,Active,2025-11-01,41.22
4,P004952,Granite Processing 4952,Technology Platform,Travel,AZ,Active,2025-12-01,40.97


In [14]:
q1_result.to_csv(os.path.join(output_path, "q1_technology_az_growth_above_20.csv"), index=False)

**Business Question 2**

Which partners are on the Strategic pricing tier?

In [15]:
q2_sql = """
SELECT
    pm.partner_id,
    pm.partner_name,
    pm.partner_status,
    pp.pricing_tier,
    pp.recommended_pricing_plan_id,
    pp.approval_status
FROM partner_master pm
JOIN partner_pricing pp
    ON pm.partner_id = pp.partner_id
WHERE pp.pricing_tier = 'STRATEGIC'
ORDER BY pm.partner_name;
"""

q2_result = run_sql(q2_sql)
q2_result.head()

,partner_id,partner_name,partner_status,pricing_tier,recommended_pricing_plan_id,approval_status
0,P000760,BluePeak Billing 0760,Active,STRATEGIC,STRATEGIC,Not Required
1,P002523,BluePeak Billing 2523,Active,STRATEGIC,STRATEGIC,Not Required
2,P004400,BluePeak Billing 4400,Active,STRATEGIC,STRATEGIC,Not Required
3,P004791,BluePeak Billing 4791,Active,STRATEGIC,STRATEGIC,Not Required
4,P001740,BluePeak Checkout 1740,Active,STRATEGIC,STRATEGIC,Not Required


In [16]:
q2_result.to_csv(os.path.join(output_path, "q2_strategic_pricing_partners.csv"), index=False)

**Business Question 3**

Which partners have pricing reviews due in the next 90 days?

In [17]:
q3_sql = """
SELECT
    pm.partner_id,
    pm.partner_name,
    pp.pricing_tier,
    pp.approval_status,
    pp.review_due_date,
    pp.contract_status
FROM partner_master pm
JOIN partner_pricing pp
    ON pm.partner_id = pp.partner_id
WHERE date(pp.review_due_date) BETWEEN date('2026-06-28') AND date('2026-09-26')
ORDER BY date(pp.review_due_date);
"""

q3_result = run_sql(q3_sql)
q3_result.head()

,partner_id,partner_name,pricing_tier,approval_status,review_due_date,contract_status
0,P000170,Keystone PayWorks 0170,LAUNCH,Not Required,2026-06-28,Review Due Soon
1,P000368,Skyline Market 0368,PLATFORM_ISV,Not Required,2026-06-28,Review Due Soon
2,P003504,Cedar PayWorks 3504,GROWTH,Not Required,2026-06-28,Review Due Soon
3,P003984,Evergreen Financial 3984,LAUNCH,Not Required,2026-06-28,Review Due Soon
4,P004509,Prairie Payments 4509,PROMO_GROWTH,Approved,2026-06-28,Review Due Soon


In [18]:
q3_result.to_csv(os.path.join(output_path, "q3_pricing_reviews_due_90_days.csv"), index=False)

**Business Question 4**

Which industry has the highest average transaction volume?

In [19]:
q4_sql = """
SELECT
    pm.industry_vertical,
    ROUND(AVG(mt.transaction_volume), 2) AS avg_transaction_volume
FROM partner_master pm
JOIN partner_metrics mt
    ON pm.partner_id = mt.partner_id
GROUP BY pm.industry_vertical
ORDER BY avg_transaction_volume DESC;
"""

q4_result = run_sql(q4_sql)
q4_result

,industry_vertical,avg_transaction_volume
0,Gaming,5018721.46
1,Travel,4521112.13
2,Fintech,4449626.97
3,Utilities,4153576.08
4,Logistics,4149628.57
5,Healthcare,3949840.39
6,Government,3896428.26
7,Hospitality,3887502.99
8,Retail,3876466.52
9,Professional Services,3814276.01


In [20]:
q4_result.to_csv(os.path.join(output_path, "q4_avg_transaction_volume_by_industry.csv"), index=False)

**Business Question 5**

Show the top 5 partners by revenue.

In [21]:
q5_sql = """
SELECT
    pm.partner_id,
    pm.partner_name,
    ROUND(SUM(mt.revenue), 2) AS total_revenue
FROM partner_master pm
JOIN partner_metrics mt
    ON pm.partner_id = mt.partner_id
GROUP BY pm.partner_id, pm.partner_name
ORDER BY total_revenue DESC
LIMIT 5;
"""

q5_result = run_sql(q5_sql)
q5_result

,partner_id,partner_name,total_revenue
0,P000610,Skyline Payments 0610,64847292.41
1,P002651,RedRock PayTech 2651,28539531.11
2,P003122,Cedar Market 3122,27485493.95
3,P004602,NorthStar Commerce 4602,26392278.10
4,P001422,Brightpath Platform 1422,25564051.57


In [22]:
q5_result.to_csv(os.path.join(output_path, "q5_top_5_partners_by_revenue.csv"), index=False)

**Question 6**

Which active partners have pricing exceptions?

In [23]:
q6_sql = """
SELECT
    pm.partner_id,
    pm.partner_name,
    pm.partner_status,
    pp.pricing_tier,
    pp.exception_flag,
    pp.exception_type,
    pp.negotiated_bps,
    pp.approval_status
FROM partner_master pm
JOIN partner_pricing pp
    ON pm.partner_id = pp.partner_id
WHERE pm.partner_status = 'Active'
  AND pp.exception_flag = 1
ORDER BY pm.partner_name;
"""

q6_result = run_sql(q6_sql)
q6_result.head()

,partner_id,partner_name,partner_status,pricing_tier,exception_flag,exception_type,negotiated_bps,approval_status
0,P000573,BluePeak Billing 0573,Active,LEGACY_STANDARD,1,Legacy plan extension,53.65,Approved
1,P000735,BluePeak Billing 0735,Active,PLATFORM_ISV,1,High-risk waiver,55.48,Approved
2,P004415,BluePeak Billing 4415,Active,GROWTH,1,Volume threshold waiver,52.99,Pending
3,P002322,BluePeak Checkout 2322,Active,LEGACY_STANDARD,1,Waived monthly minimum,67.65,Pending
4,P002996,BluePeak Checkout 2996,Active,SCALE,1,High-risk waiver,48.18,Approved


In [24]:
q6_result.to_csv(os.path.join(output_path, "q6_active_partners_with_pricing_exceptions.csv"), index=False)

**Question 7**

Which partners have missing city or state information?

In [25]:
q7_sql = """
SELECT
    partner_id,
    partner_name,
    city,
    state,
    region
FROM partner_master
WHERE city IS NULL
   OR state IS NULL
   OR TRIM(city) = ''
   OR TRIM(state) = '';
"""

q7_result = run_sql(q7_sql)
q7_result

,partner_id,partner_name,city,state,region


In [26]:
q7_result.to_csv(os.path.join(output_path, "q7_missing_demographic_information.csv"), index=False)

**Question 8**

Compare average transaction growth by pricing tier.

In [27]:
q8_sql = """
SELECT
    pp.pricing_tier,
    ROUND(AVG(mt.growth_pct), 2) AS avg_growth_pct,
    COUNT(*) AS metric_record_count
FROM partner_pricing pp
JOIN partner_metrics mt
    ON pp.partner_id = mt.partner_id
WHERE mt.growth_pct IS NOT NULL
GROUP BY pp.pricing_tier
ORDER BY avg_growth_pct DESC;
"""

q8_result = run_sql(q8_sql)
q8_result

,pricing_tier,avg_growth_pct,metric_record_count
0,PLATFORM_ISV,2.97,16399
1,LEGACY_STANDARD,2.77,2967
2,STRATEGIC,2.44,8993
3,HIGH_RISK,2.42,19803
4,SCALE,2.40,13662
5,GROWTH,2.09,24219
6,PROMO_GROWTH,2.00,943
7,LEGACY_ENTERPRISE,2.00,3197
8,LAUNCH,1.36,24817


In [28]:
q8_result.to_csv(os.path.join(output_path, "q8_avg_growth_by_pricing_tier.csv"), index=False)

**Question 9**

Which enterprise partners are in California?

In [29]:
q9_sql = """
SELECT
    partner_id,
    partner_name,
    partner_size,
    state,
    city,
    partner_status
FROM partner_master
WHERE partner_size = 'Enterprise'
  AND state = 'CA'
ORDER BY partner_name;
"""

q9_result = run_sql(q9_sql)
q9_result

,partner_id,partner_name,partner_size,state,city,partner_status
0,P001362,BluePeak Checkout 1362,Enterprise,CA,Sacramento,Active
1,P001393,BluePeak Commerce 1393,Enterprise,CA,San Diego,Dormant
2,P003724,BluePeak Connect 3724,Enterprise,CA,San Diego,Active
3,P002564,BluePeak Financial 2564,Enterprise,CA,San Jose,Active
4,P001412,BluePeak Merchant Services 1412,Enterprise,CA,San Diego,Dormant
...,...,...,...,...,...,...
140,P003128,Sunrise PayTech 3128,Enterprise,CA,San Jose,Active
141,P000622,Sunrise PayWorks 0622,Enterprise,CA,San Diego,Active
142,P001796,Sunrise Payments 1796,Enterprise,CA,Sacramento,Active
143,P001885,Sunrise Payments 1885,Enterprise,CA,Sacramento,Active


In [30]:
q9_result.to_csv(os.path.join(output_path, "q9_enterprise_partners_california.csv"), index=False)

**Question 10**

Which partners had declining transaction growth in the latest month?

In [31]:
q10_sql = """
WITH latest_month AS (
    SELECT MAX(date(transaction_month)) AS max_month
    FROM partner_metrics
)
SELECT
    pm.partner_id,
    pm.partner_name,
    mt.transaction_month,
    mt.growth_pct
FROM partner_master pm
JOIN partner_metrics mt
    ON pm.partner_id = mt.partner_id
WHERE date(mt.transaction_month) = (SELECT max_month FROM latest_month)
  AND mt.growth_pct < 0
ORDER BY mt.growth_pct ASC;
"""

q10_result = run_sql(q10_sql)
q10_result.head()

,partner_id,partner_name,transaction_month,growth_pct
0,P002649,Sunrise Billing 2649,2025-12-01,-27.27
1,P000524,BluePeak Processing 0524,2025-12-01,-26.10
2,P004263,Oakstone Payments 4263,2025-12-01,-25.77
3,P002921,RedRock Payments 2921,2025-12-01,-25.10
4,P002253,Maple Checkout 2253,2025-12-01,-24.96


In [32]:
q10_result.to_csv(os.path.join(output_path, "q10_declining_growth_latest_month.csv"), index=False)

# Step 17: Create a Benchmark Question Bank

In [33]:
benchmark_questions = pd.DataFrame({
    "question_id": [
        "Q001", "Q002", "Q003", "Q004", "Q005",
        "Q006", "Q007", "Q008", "Q009", "Q010"
    ],
    "business_question": [
        "Show me technology partners in Arizona with transaction growth above 20%.",
        "Which partners are on the Strategic pricing tier?",
        "Which partners have pricing reviews due in the next 90 days?",
        "Which industry has the highest average transaction volume?",
        "Show the top 5 partners by revenue.",
        "Which active partners have pricing exceptions?",
        "Which partners have missing city or state information?",
        "Compare average transaction growth by pricing tier.",
        "Which enterprise partners are in California?",
        "Which partners had declining transaction growth in the latest month?"
    ],
    "expected_tables": [
        "partner_master, partner_metrics",
        "partner_master, partner_pricing",
        "partner_master, partner_pricing",
        "partner_master, partner_metrics",
        "partner_master, partner_metrics",
        "partner_master, partner_pricing",
        "partner_master",
        "partner_pricing, partner_metrics",
        "partner_master",
        "partner_master, partner_metrics"
    ],
    "expected_fields": [
        "partner_type, state, growth_pct",
        "partner_name, pricing_tier",
        "partner_name, review_due_date, contract_status",
        "industry_vertical, transaction_volume",
        "partner_name, revenue",
        "partner_status, exception_flag, exception_type",
        "city, state",
        "pricing_tier, growth_pct",
        "partner_size, state",
        "transaction_month, growth_pct"
    ],
    "answer_type": [
        "Filtered list",
        "Filtered list",
        "Filtered list",
        "Ranking summary",
        "Top 5 ranking",
        "Filtered list",
        "Data quality list",
        "Comparison summary",
        "Filtered list",
        "Filtered list"
    ],
    "citation_required": [
        "Yes", "Yes", "Yes", "Yes", "Yes",
        "Yes", "Yes", "Yes", "Yes", "Yes"
    ]
})

benchmark_questions

,question_id,business_question,expected_tables,expected_fields,answer_type,citation_required
0,Q001,Show me technology partners in Arizona with tr...,"partner_master, partner_metrics","partner_type, state, growth_pct",Filtered list,Yes
1,Q002,Which partners are on the Strategic pricing tier?,"partner_master, partner_pricing","partner_name, pricing_tier",Filtered list,Yes
2,Q003,Which partners have pricing reviews due in the...,"partner_master, partner_pricing","partner_name, review_due_date, contract_status",Filtered list,Yes
3,Q004,Which industry has the highest average transac...,"partner_master, partner_metrics","industry_vertical, transaction_volume",Ranking summary,Yes
4,Q005,Show the top 5 partners by revenue.,"partner_master, partner_metrics","partner_name, revenue",Top 5 ranking,Yes
5,Q006,Which active partners have pricing exceptions?,"partner_master, partner_pricing","partner_status, exception_flag, exception_type",Filtered list,Yes
6,Q007,Which partners have missing city or state info...,partner_master,"city, state",Data quality list,Yes
7,Q008,Compare average transaction growth by pricing ...,"partner_pricing, partner_metrics","pricing_tier, growth_pct",Comparison summary,Yes
8,Q009,Which enterprise partners are in California?,partner_master,"partner_size, state",Filtered list,Yes
9,Q010,Which partners had declining transaction growt...,"partner_master, partner_metrics","transaction_month, growth_pct",Filtered list,Yes


In [34]:
benchmark_questions.to_csv(
    os.path.join(output_path, "phase4_benchmark_questions.csv"),
    index=False
)

# Step 18: Save SQL Queries into a File

In [35]:
sql_queries = {
    "Q001": q1_sql,
    "Q002": q2_sql,
    "Q003": q3_sql,
    "Q004": q4_sql,
    "Q005": q5_sql,
    "Q006": q6_sql,
    "Q007": q7_sql,
    "Q008": q8_sql,
    "Q009": q9_sql,
    "Q010": q10_sql
}

sql_query_df = pd.DataFrame({
    "question_id": list(sql_queries.keys()),
    "sql_query": list(sql_queries.values())
})

sql_query_df.to_csv(
    os.path.join(output_path, "phase4_baseline_sql_queries.csv"),
    index=False
)

sql_query_df

,question_id,sql_query
0,Q001,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
1,Q002,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
2,Q003,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
3,Q004,"\nSELECT\n pm.industry_vertical,\n ROUND..."
4,Q005,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
5,Q006,"\nSELECT\n pm.partner_id,\n pm.partner_n..."
6,Q007,"\nSELECT\n partner_id,\n partner_name,\n..."
7,Q008,"\nSELECT\n pp.pricing_tier,\n ROUND(AVG(..."
8,Q009,"\nSELECT\n partner_id,\n partner_name,\n..."
9,Q010,\nWITH latest_month AS (\n SELECT MAX(date(...


# Step 19: Create a Query Result Summary

In [36]:
query_result_summary = pd.DataFrame({
    "question_id": [
        "Q001", "Q002", "Q003", "Q004", "Q005",
        "Q006", "Q007", "Q008", "Q009", "Q010"
    ],
    "result_row_count": [
        len(q1_result),
        len(q2_result),
        len(q3_result),
        len(q4_result),
        len(q5_result),
        len(q6_result),
        len(q7_result),
        len(q8_result),
        len(q9_result),
        len(q10_result)
    ],
    "output_file": [
        "q1_technology_az_growth_above_20.csv",
        "q2_strategic_pricing_partners.csv",
        "q3_pricing_reviews_due_90_days.csv",
        "q4_avg_transaction_volume_by_industry.csv",
        "q5_top_5_partners_by_revenue.csv",
        "q6_active_partners_with_pricing_exceptions.csv",
        "q7_missing_demographic_information.csv",
        "q8_avg_growth_by_pricing_tier.csv",
        "q9_enterprise_partners_california.csv",
        "q10_declining_growth_latest_month.csv"
    ]
})

query_result_summary

,question_id,result_row_count,output_file
0,Q001,85,q1_technology_az_growth_above_20.csv
1,Q002,391,q2_strategic_pricing_partners.csv
2,Q003,361,q3_pricing_reviews_due_90_days.csv
3,Q004,13,q4_avg_transaction_volume_by_industry.csv
4,Q005,5,q5_top_5_partners_by_revenue.csv
5,Q006,681,q6_active_partners_with_pricing_exceptions.csv
6,Q007,0,q7_missing_demographic_information.csv
7,Q008,9,q8_avg_growth_by_pricing_tier.csv
8,Q009,145,q9_enterprise_partners_california.csv
9,Q010,1298,q10_declining_growth_latest_month.csv


In [37]:
query_result_summary.to_csv(
    os.path.join(output_path, "phase4_query_result_summary.csv"),
    index=False
)

# Step 20: Save Everything into One Excel Workbook

In [38]:
excel_output = os.path.join(output_path, "partnerlens_phase4_sql_baseline_results.xlsx")

with pd.ExcelWriter(excel_output, engine="openpyxl") as writer:
    row_counts.to_excel(writer, sheet_name="table_row_counts", index=False)
    benchmark_questions.to_excel(writer, sheet_name="benchmark_questions", index=False)
    sql_query_df.to_excel(writer, sheet_name="baseline_sql_queries", index=False)
    query_result_summary.to_excel(writer, sheet_name="query_result_summary", index=False)
    q1_result.to_excel(writer, sheet_name="q1_result", index=False)
    q2_result.to_excel(writer, sheet_name="q2_result", index=False)
    q3_result.to_excel(writer, sheet_name="q3_result", index=False)
    q4_result.to_excel(writer, sheet_name="q4_result", index=False)
    q5_result.to_excel(writer, sheet_name="q5_result", index=False)
    q6_result.to_excel(writer, sheet_name="q6_result", index=False)
    q7_result.to_excel(writer, sheet_name="q7_result", index=False)
    q8_result.to_excel(writer, sheet_name="q8_result", index=False)
    q9_result.to_excel(writer, sheet_name="q9_result", index=False)
    q10_result.to_excel(writer, sheet_name="q10_result", index=False)

print("Excel file saved to:", excel_output)

Excel file saved to: /content/drive/MyDrive/Capstone Project/partnerlens_synthetic_data_package/partnerlens_data_package/outputs/phase4_sql_baseline/partnerlens_phase4_sql_baseline_results.xlsx


# Step 21: Close the SQLite Connection

In [39]:
conn.close()
print("SQLite connection closed.")

SQLite connection closed.
